# SIRCH - Entrainement local augmente

Notebook d'entrainement du Systeme Intelligent de Reconnaissance de Comportements Humains.

Les donnees sont lues depuis `datasets\train_augmente`. Google Drive n'est utilise que pour copier le modele final `sirch_model_augmente.h5`.

In [ ]:
# ============================================================
# CELLULE 1 - Chemins locaux SIRCH
# ============================================================
from pathlib import Path
import os

PROJECT_DATASETS_ROOT = Path(os.environ.get('SIRCH_PROJECT_DATASETS_DIR', r'C:\Users\djtra\Documents\Codex\2026-05-19\files-mentioned-by-the-user-sirch\SIRCH\datasets'))
AUGMENTED_DATASET_DIR = Path(os.environ.get('SIRCH_AUGMENTED_DATASET_DIR', str(PROJECT_DATASETS_ROOT / 'train_augmente')))
DATASETS_ROOT = AUGMENTED_DATASET_DIR
LOCAL_MODEL_DIR = Path(os.environ.get('SIRCH_LOCAL_MODEL_DIR', r'C:\SIRCH_ENV\models'))
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Datasets augmentes : {DATASETS_ROOT}')
print(f'Dossier modele local : {LOCAL_MODEL_DIR}')

if not DATASETS_ROOT.exists():
    raise RuntimeError(r'Dossier datasets augmentes introuvable. Verifie datasets\train_augmente avant de lancer ce notebook.')

In [ ]:
# ============================================================
# CELLULE 2 - Chemin Drive pour le modele final
# ============================================================
# Les datasets restent locaux. Ce chemin sert seulement a copier sirch_model_augmente.h5
# vers Google Drive a la fin, si Google Drive Desktop est disponible sur Windows.
# Exemple possible plus tard : r'G:\My Drive\SIRCH\models\sirch_model_augmente.h5'
DRIVE_MODEL_OUTPUT = os.environ.get('SIRCH_DRIVE_MODEL_AUGMENTE_OUTPUT', r'')

if DRIVE_MODEL_OUTPUT:
    Path(DRIVE_MODEL_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    print(f'Modele final Drive : {DRIVE_MODEL_OUTPUT}')
else:
    print('Aucun chemin Drive local configure pour le modele final.')
    print('On configurera SIRCH_DRIVE_MODEL_AUGMENTE_OUTPUT au moment de lancer l entrainement augmente.')

In [ ]:
# ============================================================
# CELLULE 3 - Kaggle est configure par le script local
# ============================================================
print(r'Ne pas uploader kaggle.json dans ce notebook.')
print(r'Le script C:\SIRCH_ENV\download_datasets.py utilisera ton kaggle.json local.')

In [ ]:
# ============================================================
# CELLULE 4 - RLVS se telecharge en local
# ============================================================
print(r'RLVS ne se telecharge plus vers Google Drive depuis ce notebook.')
print(r'Lance dans l invite de commande :')
print(r'C:\SIRCH_ENV\Scripts\python.exe C:\SIRCH_ENV\download_datasets.py --only rlvs')

In [ ]:
# ============================================================
# CELLULE 5 - RWF-2000 se telecharge en local
# ============================================================
print(r'RWF-2000 ne se telecharge plus vers Google Drive depuis ce notebook.')
print(r'Lance dans l invite de commande :')
print(r'C:\SIRCH_ENV\Scripts\python.exe C:\SIRCH_ENV\download_datasets.py --only rwf')

In [ ]:
# ============================================================
# CELLULE 6 - Parametres SIRCH
# ============================================================
import glob
import random
import time
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

N_FRAMES = 20
IMG_SIZE = 224
LSTM_UNITS = 256
DROPOUT = 0.5

BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
OPTIMIZER = 'adam'
LOSS = 'binary_crossentropy'

if 'DATASETS_ROOT' not in globals():
    PROJECT_DATASETS_ROOT = Path(os.environ.get('SIRCH_PROJECT_DATASETS_DIR', r'C:\Users\djtra\Documents\Codex\2026-05-19\files-mentioned-by-the-user-sirch\SIRCH\datasets'))
    DATASETS_ROOT = Path(os.environ.get('SIRCH_AUGMENTED_DATASET_DIR', str(PROJECT_DATASETS_ROOT / 'train_augmente')))
if 'LOCAL_MODEL_DIR' not in globals():
    LOCAL_MODEL_DIR = Path(os.environ.get('SIRCH_LOCAL_MODEL_DIR', r'C:\SIRCH_ENV\models'))
if 'DRIVE_MODEL_OUTPUT' not in globals():
    DRIVE_MODEL_OUTPUT = os.environ.get('SIRCH_DRIVE_MODEL_AUGMENTE_OUTPUT', r'')

LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIRS = [str(DATASETS_ROOT)]
CHECKPOINT_DIR = LOCAL_MODEL_DIR / 'checkpoints_augmente'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EPOCH_CHECKPOINT_PATTERN = str(CHECKPOINT_DIR / 'sirch_augmente_weights_epoch_{epoch:03d}.weights.h5')
TRAINING_LOG = str(LOCAL_MODEL_DIR / 'training_log_augmente.csv')
LOCAL_MODEL_OUTPUT = str(LOCAL_MODEL_DIR / 'sirch_model_augmente.h5')
MODEL_OUTPUT = DRIVE_MODEL_OUTPUT or LOCAL_MODEL_OUTPUT

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print('Parametres SIRCH charges.')
print(f'Datasets : {DATASETS_DIRS}')
print(f'Checkpoints : {CHECKPOINT_DIR}')
print(f'Modele final : {MODEL_OUTPUT}')

In [ ]:
# ============================================================
# CELLULE 7 - Indexer les videos du dataset augmente
# ============================================================
VIDEO_EXTENSIONS = ('*.mp4', '*.avi', '*.mov', '*.mkv')

def collect_from_folder(folder, label):
    files = []
    for ext in VIDEO_EXTENSIONS:
        files.extend(glob.glob(os.path.join(folder, '**', ext), recursive=True))
    return [(path, label) for path in files]

def collect_dataset(root):
    samples = []
    label_rules = {
        1: ['Violence', 'Fight'],
        0: ['NonViolence', 'NonFight', 'Non-Violence', 'Non Violence']
    }
    for label, names in label_rules.items():
        for name in names:
            for folder in glob.glob(os.path.join(root, '**', name), recursive=True):
                if os.path.isdir(folder):
                    samples.extend(collect_from_folder(folder, label))
    unique = {}
    for path, label in samples:
        unique[path] = label
    return list(unique.items())

samples = []
for dataset_dir in DATASETS_DIRS:
    samples.extend(collect_dataset(dataset_dir))
samples = list(dict(samples).items())
random.shuffle(samples)
labels = [label for _, label in samples]
print(f'Videos trouvees : {len(samples)}')
print(f'Violence : {sum(labels)} | Non-violence : {len(labels) - sum(labels)}')
if not samples:
    raise RuntimeError(r'Aucune video trouvee. Verifie datasets\train_augmente avant de lancer ce notebook.')

In [ ]:
# ============================================================
# CELLULE 8 â€” RÃ©partition 70% / 15% / 15%
# ============================================================
paths = [path for path, _ in samples]
labels = [label for _, label in samples]

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels, test_size=0.30, random_state=42, stratify=labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print(f'Train : {len(train_paths)} clips')
print(f'Validation : {len(val_paths)} clips')
print(f'Test : {len(test_paths)} clips')

In [ ]:
# ============================================================
# CELLULE 9 â€” GÃ©nÃ©rateur vidÃ©o mÃ©moire-efficace
# ============================================================
class VideoSequence(tf.keras.utils.Sequence):
    def __init__(self, video_paths, labels, batch_size=BATCH_SIZE, shuffle=True):
        self.video_paths = list(video_paths)
        self.labels = np.array(labels, dtype=np.float32)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.video_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.video_paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x = np.zeros((len(batch_indices), N_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        batch_y = self.labels[batch_indices]
        for i, sample_idx in enumerate(batch_indices):
            batch_x[i] = load_video_frames(self.video_paths[sample_idx])
        return batch_x, batch_y

def load_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return np.zeros((N_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)

    frame_indices = np.linspace(0, max(total - 1, 0), N_FRAMES).astype(int)
    frames = []
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frames.append(frame.astype(np.float32))
    cap.release()
    frames = np.stack(frames, axis=0)
    return tf.keras.applications.efficientnet.preprocess_input(frames)

train_gen = VideoSequence(train_paths, train_labels, shuffle=True)
val_gen = VideoSequence(val_paths, val_labels, shuffle=False)
test_gen = VideoSequence(test_paths, test_labels, shuffle=False)
print('âœ… GÃ©nÃ©rateurs prÃªts.')

In [ ]:
# ============================================================
# CELLULE 10 â€” Architecture EfficientNetB0 + LSTM
# ============================================================
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

sequence_input = layers.Input(shape=(N_FRAMES, IMG_SIZE, IMG_SIZE, 3))
features = layers.TimeDistributed(base_model)(sequence_input)
x = layers.LSTM(LSTM_UNITS, return_sequences=False)(features)
x = layers.Dropout(DROPOUT)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs=sequence_input, outputs=output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=LOSS,
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

In [ ]:
# ============================================================
# CELLULE 11 - Entrainement avec reprise automatique
# ============================================================
import csv
import re

def checkpoint_epoch(path):
    match = re.search(r'sirch_augmente_weights_epoch_(\d+)(?:\.weights)?\.h5$', path)
    return int(match.group(1)) if match else -1

checkpoint_files = sorted(
    set(glob.glob(str(CHECKPOINT_DIR / 'sirch_augmente_weights_epoch_*.h5'))),
    key=checkpoint_epoch
)
initial_epoch = 0
if checkpoint_files:
    latest_checkpoint = checkpoint_files[-1]
    match = re.search(r'sirch_augmente_weights_epoch_(\d+)(?:\.weights)?\.h5$', latest_checkpoint)
    if match:
        initial_epoch = int(match.group(1))
    print(f'Reprise depuis les poids du checkpoint : {latest_checkpoint}')
    model.load_weights(latest_checkpoint)
else:
    print('Aucun checkpoint trouve. Entrainement depuis le debut.')

best_val_loss = None
if Path(TRAINING_LOG).exists() and Path(LOCAL_MODEL_OUTPUT).exists():
    with open(TRAINING_LOG, newline='') as file:
        for row in csv.DictReader(file):
            value = row.get('val_loss')
            if value not in (None, ''):
                value = float(value)
                best_val_loss = value if best_val_loss is None else min(best_val_loss, value)

epoch_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    EPOCH_CHECKPOINT_PATTERN,
    save_best_only=False,
    save_weights_only=True
)
best_model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    LOCAL_MODEL_OUTPUT,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    save_weights_only=False,
    initial_value_threshold=best_val_loss
)
csv_logger = tf.keras.callbacks.CSVLogger(TRAINING_LOG, append=True)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

if initial_epoch >= EPOCHS:
    print(f'Entrainement deja arrive a {initial_epoch} epochs sur {EPOCHS}.')
else:
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch,
        epochs=EPOCHS,
        callbacks=[epoch_checkpoint, best_model_checkpoint, csv_logger, early_stop, reduce_lr]
    )

if not Path(LOCAL_MODEL_OUTPUT).exists():
    model.save(LOCAL_MODEL_OUTPUT)
print(f'Meilleurs poids du modele final local : {LOCAL_MODEL_OUTPUT}')

if DRIVE_MODEL_OUTPUT:
    import shutil
    Path(DRIVE_MODEL_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(LOCAL_MODEL_OUTPUT, DRIVE_MODEL_OUTPUT)
    print(f'Modele final copie sur Drive : {DRIVE_MODEL_OUTPUT}')
else:
    print('Chemin Drive non configure. Le modele final reste local pour l instant.')

In [ ]:
# ============================================================
# CELLULE 12 â€” Ã‰valuation obligatoire
# ============================================================
start = time.time()
scores = model.predict(test_gen).ravel()
elapsed = time.time() - start
preds = (scores >= 0.5).astype(int)
truth = np.array(test_labels, dtype=int)

accuracy = accuracy_score(truth, preds)
precision = precision_score(truth, preds, zero_division=0)
recall = recall_score(truth, preds, zero_division=0)
f1 = f1_score(truth, preds, zero_division=0)
ms_per_frame = (elapsed / max(len(test_paths) * N_FRAMES, 1)) * 1000
cm = confusion_matrix(truth, preds)

print(f'Accuracy : {accuracy:.4f}')
print(f'PrÃ©cision : {precision:.4f}')
print(f'Rappel : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'Temps d\'infÃ©rence moyen : {ms_per_frame:.2f} ms/frame')
print('Matrice de confusion :')
print(cm)
print('RÃ©fÃ©rence Ã  battre : F1 = 86.39% (Abdullah et al. 2023 sur RLVS seul)')